# Reducir a 30 variables o dimensiones meteorológicas y epidemiológicas

Aquí tienes el script de Python modificado para expandir la selección a un total de **30 variables predictoras** bajo la misma metodología rigurosa de Spearman (alta correlación con el objetivo y baja correlación entre sí para mitigar la multicolinealidad).

El script se ha configurado con las rutas exactas de entrada y salida que has especificado para cada uno de los archivos correspondientes.


In [3]:
import pandas as pd
import numpy as np
import os

# =============================================================================
# 1. DEFINICIÓN DE RUTAS Y CONFIGURACIÓN
# =============================================================================
ruta_entrada = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx"
carpeta_datos_procesados = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\2_procesados"
carpeta_resultados = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\3_resultados"

# Asegurar que las carpetas de destino existan
os.makedirs(carpeta_datos_procesados, exist_ok=True)
os.makedirs(carpeta_resultados, exist_ok=True)

ruta_dataset_final = os.path.join(carpeta_datos_procesados, "2_meteo_epi_spearman_meteo_epi_30.xlsx")
ruta_reporte_variables = os.path.join(carpeta_resultados, "2_lista_variables_seleccionadas_spearman_30.xlsx")

print("Cargando el archivo de datos original con variables rezagadas...")
df = pd.read_excel(ruta_entrada)

# Asegurar orden cronológico mediante el índice temporal
df['fecha'] = pd.to_datetime(df['fecha'])
df.set_index('fecha', inplace=True)
df = df.sort_index()

# =============================================================================
# 2. METODOLOGÍA DE REDUCCIÓN DIMENSIONAL (SPEARMAN)
# =============================================================================
target = 'casos_dengue'

# Excluir los identificadores de tiempo y la variable objetivo de las características candidatas
columnas_obligatorias = ['año', 'semana_epi']
candidatas = [col for col in df.columns if col not in [target] + columnas_obligatorias]

print("Calculando la matriz de correlación de Spearman...")
matriz_corr = df[[target] + candidatas].corr(method='spearman')

# Obtener la correlación absoluta con la variable objetivo y ordenar de mayor a menor impacto
corr_con_objetivo = matriz_corr[target].drop(target).abs().sort_values(ascending=False)

# Algoritmo de filtrado: Alta correlación con objetivo, baja correlación entre sí
variables_seleccionadas = []
umbral_multicolinealidad = 0.70  # Coeficiente máximo permitido entre variables predictoras

for var in corr_con_objetivo.index:
    if len(variables_seleccionadas) >= 30:  # Detenerse al alcanzar exactamente 30 variables
        break
        
    # Verificar si esta variable es redundante con alguna de las que ya seleccionamos
    es_redundante = False
    for var_sel in variables_seleccionadas:
        if abs(matriz_corr.loc[var, var_sel]) > umbral_multicolinealidad:
            es_redundante = True
            break
            
    if not es_redundante:
        variables_seleccionadas.append(var)

# =============================================================================
# 3. GENERACIÓN Y EXPORTACIÓN DEL REPORTE DE VARIABLES (HACIA \3_resultados)
# =============================================================================
# Obtener los coeficientes reales con su signo original para el reporte final
tabla_reporte = pd.DataFrame({
    'Variable': variables_seleccionadas,
    'Correlacion_Spearman': [matriz_corr.loc[v, target] for v in variables_seleccionadas],
    'Correlacion_Absoluta': [corr_con_objetivo[v] for v in variables_seleccionadas]
})

print(f"Guardando la lista de variables seleccionadas en:\n-> {ruta_reporte_variables}")
tabla_reporte.to_excel(ruta_reporte_variables, index=False)

# Imprimir en consola para validación rápida en pantalla
print("\n" + "="*60)
print("TOP 30 VARIABLES SELECCIONADAS (MÁS INDEPENDIENTES ENTRE SÍ):")
print("="*60)
for idx, fila in tabla_reporte.iterrows():
    print(f"{idx+1:02d}. {fila['Variable']:<30} | Spearman: {fila['Correlacion_Spearman']:.4f}")
print("="*60)

# =============================================================================
# 4. CONSTRUCCIÓN Y GUARDADO DEL DATASET PROCESADO (HACIA \2_procesados)
# =============================================================================
# Combinar identificadores obligatorios, variable objetivo y los 30 predictores óptimos
columnas_finales = columnas_obligatorias + [target] + variables_seleccionadas
df_final = df[columnas_finales].copy()

print(f"\nGuardando el dataset final de 30 variables en:\n-> {ruta_dataset_final}")
# Se resetea el índice para que la columna 'fecha' se escriba correctamente en el archivo Excel
df_final.reset_index().to_excel(ruta_dataset_final, index=False)

print("\n=== PROCESO COMPLETADO EXITOSAMENTE ===")
print(f"Dimensiones del set procesado: {df_final.shape[0]} semanas x {df_final.shape[1]} columnas.")


Cargando el archivo de datos original con variables rezagadas...
Calculando la matriz de correlación de Spearman...
Guardando la lista de variables seleccionadas en:
-> C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\3_resultados\2_lista_variables_seleccionadas_spearman_30.xlsx

TOP 30 VARIABLES SELECCIONADAS (MÁS INDEPENDIENTES ENTRE SÍ):
01. casos_dengue_lag_1             | Spearman: 0.9245
02. hum_esp_lag_9                  | Spearman: 0.5529
03. hum_esp_lag_3                  | Spearman: 0.5398
04. hum_rel_lag_11                 | Spearman: 0.4012
05. vel_vi_max_lag_4               | Spearman: 0.3501
06. dias_lluvia_lag_10             | Spearman: 0.3392
07. dias_lluvia_lag_9              | Spearman: 0.3372
08. dias_lluvia_lag_7              | Spearman: 0.3327
09. vel_vi_max_lag_7               | Spearman: 0.3314
10. dias_lluvia_lag_5              | Spearman: 0.3306
11. dias_lluvia_lag_4              | Spearman: 0.3287


# Cambios clave aplicados en este script:

1. **`len(variables_seleccionadas) >= 30`**: Se expandió el límite de detención automática del algoritmo codicioso para recolectar el conjunto completo de 30 variables.
2. **`ruta_reporte_variables`**: Ahora apunta dinámicamente y de forma estricta hacia la subcarpeta `\3_resultados` guardando el reporte bajo el nombre `lista_variables_seleccionadas_spearman_30.xlsx`.
3. **`ruta_dataset_final`**: Apunta directamente a la subcarpeta `\2_datos\2_procesados` guardando el nuevo dataset bajo el nombre corporativo `2_meteo_epi_rezagos_spearman_30.xlsx`.